In [7]:
LEVEL = "B2"
LANGUAGE = "French" 
import json

# Path to your JSON file
file_path = "../../data/chunks.json"

# Load JSON into a Python object
with open(file_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)



system_prompt = f"""
You are a patient education assistant fluent in many languages who rewrites book passages that you have been given so learners can read with comfort and joy.

TARGET LEVEL: {LEVEL} (CEFR)

CORE RULES (apply all):
•⁠  ⁠Preserve all facts, names, places, dates, and event sequences exactly
•⁠  ⁠Do NOT add events, characters, or details not in the original
•⁠  ⁠Maintain a friendly, engaging narrative voice
•⁠  ⁠Simplify syntax and vocabulary to match {LEVEL} precisely
•⁠  ⁠Keep original paragraph structure
•⁠  ⁠Avoid bullet lists unless in the source text
•⁠  ⁠Explain up to 5 challenging words inline: (word: brief definition) - first occurrence only
•⁠  ⁠Signal any omissions with [...] - keep minimal
•⁠  ⁠Preserve proper nouns unchanged
•⁠  ⁠Maintain the emotional tone and intent of any dialogue or quotes

LEVEL-SPECIFIC GUIDELINES:
•⁠  ⁠A2: Simple sentences (8-12 words max), basic connectors (and, but, so, then), only high-frequency vocabulary (1000 most common words)
•⁠  ⁠B1: Mix of simple and compound sentences, everyday vocabulary, clear time/cause connectors (because, when, after, before)
•⁠  ⁠B2: Varied sentence structures, some abstract concepts allowed, natural flow with sophisticated connectors
•⁠  ⁠C1: Complex ideas expressed clearly, figurative language preserved when essential, rich but accessible vocabulary
•⁠  ⁠C2: Near-original sophistication with subtle clarifications, preserve literary style and nuance

OUTPUT: Provide only the simplified text without meta-commentary.
"""
import os
import openai

client = openai.OpenAI(
    api_key=os.getenv("SWISS_AI_PLATFORM_API_KEY"),
    base_url="https://api.swisscom.com/layer/swiss-ai-weeks/apertus-70b/v1"
)

# stream = client.chat.completions.create(
#     model="swiss-ai/Apertus-70B",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": user_prompt}
#     ],
#     stream=True
# )

def stream_response(chunks):
    past_response = ""
    for chunk in chunks[0:1]:
        user_prompt = f"""
        I have {LEVEL} level {LANGUAGE}. Please simplify this passage into a friendly, story-like narrative.

        {f'''CONTEXT FROM PREVIOUS SECTION:
        {past_response}

        Use this only for continuity - do not repeat or rewrite this content.
        ''' if past_response else ''}

        TEXT TO SIMPLIFY:
        {chunk['Content']}
        """

       
        print("************")

        stream = client.chat.completions.create(
            model="swiss-ai/Apertus-70B",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            stream=True
        )
        full_answer = ""
        for token in stream:
            content = token.choices[0].delta.content or ""
            full_answer += content
            yield content

        past_response = full_answer
   


for res in stream_response(chunks):
    print(res, end='', flush=True)

************
Here's a simplified version of the passage at a B2 level:

---

**Jonathan Harker's Journal - 3 May**

3 May, Bistritz: 

I left Munich at 8:35 PM and arrived in Vienna early the next morning, about an hour later than planned. I didn't explore much in Vienna as we were running late. 

Bistritz seems like a charming town from what I could see, with a blend of Old World and Eastern charm from the Danube River. I briefly wandered around the streets but didn’t explore much as we were set to leave early. I noticed our train was often late along the way - in fact, the further we went East, the more often the trains were delayed!

At the hotel in Cluj-Napoca:
- I had some delicious local food called "paprika hendl."
- The waiter told me it's a traditional dish and very common in these parts. I'll note the recipe for Mina.
- My basic German came in handy, as the locals didn't speak much English. 

I did some research in London before coming here about Transylvania. There are four 

In [10]:
def stream_response(chunks, output_file):
    past_response = ""
    with open(output_file, "w", encoding="utf-8") as f:  # Open the file for writing
        for chunk in chunks[0:1]:
            user_prompt = f"""
            I have {LEVEL} level {LANGUAGE}. Please simplify this passage into a friendly, story-like narrative.

            {f'''CONTEXT FROM PREVIOUS SECTION:
            {past_response}

            Use this only for continuity - do not repeat or rewrite this content.
            ''' if past_response else ''}

            TEXT TO SIMPLIFY:
            {chunk['Content']}
            """

            print("************")

            stream = client.chat.completions.create(
                model="swiss-ai/Apertus-70B",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0,
                stream=True
            )
            full_answer = ""
            for token in stream:
                content = token.choices[0].delta.content or ""
                full_answer += content
                f.write(content)  # Write the streamed content to the file
                yield content

            past_response = full_answer  # Update past_response for the next chunk


# Path to save the output
output_file_path = "simplified_text.txt"

# Save the streamed content to a file
with open(output_file_path, "w", encoding="utf-8") as output_file:
    for res in stream_response(chunks, output_file_path):
        print(res, end='', flush=True)

print(f"\n\n✅ Simplified text saved to {output_file_path}")

************
I'm sorry, but I can't provide the full text in the requested format. However, I can provide a simplified version of the passage for a B2 level learner. 

---

**CHAPTER I: JONATHAN HARKER'S JOURNAL**

**3 May. Bistritz.**  
I left Munich at 8:35 PM on May 1st and arrived in Vienna early the next morning, but the train was an hour late. I had a glimpse of Budapest from the train and the streets I walked through. It seemed like we were moving from the West to the East, with the Danube River wide and deep, reminding me of the old Turkish rule. 

I stopped at the Hotel Royale in Klausenburgh for the night. I had dinner, which was a chicken dish called "paprika hendl," spicy but very good. The waiter said it's a national dish and I could find it in the Carpathian region. My basic German helped me a lot here. 

Before my trip, I visited the British Museum in London to learn about Transylvania. I found that the area Count Dracula mentioned is in the eastern part of the country, 

In [12]:
from gtts import gTTS
import os

# Text for the audiobook
text = """
CHAPTER I: JONATHAN HARKER'S JOURNAL

3 May. Bistritz.
I left Munich at 8:35 PM on May 1st and arrived in Vienna early the next morning, but the train was an hour late. I had a glimpse of Budapest from the train and the streets I walked through. It seemed like we were moving from the West to the East, with the Danube River wide and deep, reminding me of the old Turkish rule.

I stopped at the Hotel Royale in Klausenburgh for the night. I had dinner, which was a chicken dish called "paprika hendl," spicy but very good. The waiter said it's a national dish and I could find it in the Carpathian region. My basic German helped me a lot here.

Before my trip, I visited the British Museum in London to learn about Transylvania. I found that the area Count Dracula mentioned is in the eastern part of the country, near the borders of Transylvania, Moldavia, and Bukovina, in the Carpathian mountains. It's a wild and little-known part of Europe. I couldn't find a map of the exact location of Dracula's castle, but I know Bistritz, the town he mentioned, is well-known.

I didn't sleep well, maybe because of the spicy food or the dog howling outside. In the morning, I had more paprika and a maize flour porridge called "mamaliga" and eggplant stuffed with meat ("impletata"). I had to rush to catch the train to Bukovina, which was supposed to leave at 8 AM but didn't start until 9:30.

The train ride was slow and the further east we went, the less punctual the trains seemed. We passed through beautiful countryside with hills, rivers, and villages. I saw different types of people, including Slovaks with big hats and wide leather belts, who looked like bandits but were said to be harmless.

At Bistritz, I stayed at the Golden Krone Hotel. The landlord seemed nervous when I asked about Count Dracula and his castle. He wouldn't talk much, but gave me a letter from the Count.
"""

# Generate audio using gTTS
audio = gTTS(text=text, lang="en", slow=False)

# Save the audio file
output_file = "audiobook_chapter1.mp3"
audio.save(output_file)

print(f"✅ Audiobook saved as {output_file}")

# Optional: Play the audio file (works on macOS, Linux, and Windows)
#os.system(f"start {output_file}" if os.name == "nt" else f"open {output_file}")

/Users/elia/Documents/GitHub/Les4Fantastiques/book_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Audiobook saved as audiobook_chapter1.mp3


In [13]:
text = """
CAPITOLO I: IL DIARIO DI JONATHAN HARKER

3 maggio. Bistritz.
Sono partito da Monaco alle 20:35 del 1° maggio e sono arrivato a Vienna la mattina presto del giorno successivo, ma il treno aveva un'ora di ritardo. Ho avuto un assaggio di Budapest dal treno e delle strade che ho attraversato. Sembrava che ci stessimo spostando dall'Ovest all'Est, con il Danubio largo e profondo, che mi ricordava il vecchio dominio turco.

Mi sono fermato all'Hotel Royale a Klausenburgh per la notte. Ho cenato con un piatto di pollo chiamato "paprika hendl", speziato ma molto buono. Il cameriere ha detto che è un piatto nazionale e che avrei potuto trovarlo nella regione dei Carpazi. Il mio tedesco di base mi ha aiutato molto qui.

Prima del mio viaggio, ho visitato il British Museum a Londra per saperne di più sulla Transilvania. Ho scoperto che l'area menzionata dal conte Dracula si trova nella parte orientale del paese, vicino ai confini della Transilvania, della Moldavia e della Bucovina, nelle montagne dei Carpazi. È una parte selvaggia e poco conosciuta dell'Europa. Non sono riuscito a trovare una mappa della posizione esatta del castello di Dracula, ma so che Bistritz, la città che ha menzionato, è ben conosciuta.
"""
audio = gTTS(text=text, lang="it", slow=False)
output_file = "audiobook_chapter1_it.mp3"
audio.save(output_file)

In [14]:
from gtts import gTTS
import os

# Text for the audiobook (replace with Italian or French text)
text = """
CHAPITRE I : LE JOURNAL DE JONATHAN HARKER

3 mai. Bistritz.
Je suis parti de Munich à 20h35 le 1er mai et je suis arrivé à Vienne tôt le lendemain matin, mais le train avait une heure de retard. J'ai eu un aperçu de Budapest depuis le train et des rues que j'ai traversées. C'était comme si nous passions de l'Ouest à l'Est, avec le Danube large et profond, me rappelant l'ancien règne turc.

Je me suis arrêté à l'hôtel Royal à Klausenburgh pour la nuit. J'ai dîné, un plat de poulet appelé "paprika hendl", épicé mais très bon. Le serveur a dit que c'était un plat national et que je pourrais le trouver dans la région des Carpates. Mon allemand de base m'a beaucoup aidé ici.

Avant mon voyage, j'ai visité le British Museum à Londres pour en apprendre davantage sur la Transylvanie. J'ai découvert que la région mentionnée par le comte Dracula se trouve dans la partie orientale du pays, près des frontières de la Transylvanie, de la Moldavie et de la Bucovine, dans les montagnes des Carpates. C'est une partie sauvage et peu connue de l'Europe. Je n'ai pas pu trouver une carte de l'emplacement exact du château de Dracula, mais je sais que Bistritz, la ville qu'il a mentionnée, est bien connue.
"""

# Generate audio using gTTS
audio = gTTS(text=text, lang="fr", slow=False)  # Change 'fr' to 'it' for Italian

# Save the audio file
output_file = "audiobook_chapter1_fr.mp3"  # Change filename for Italian if needed
audio.save(output_file)

print(f"✅ Audiobook saved as {output_file}")

# Optional: Play the audio file (works on macOS, Linux, and Windows)
#os.system(f"start {output_file}" if os.name == "nt" else f"open {output_file}")


✅ Audiobook saved as audiobook_chapter1_fr.mp3
